# Strict-overshoot bootstrap CIs

Paired, stratified test-set bootstrap over the per-item probe scores saved in each
trajectory's `sweep_results.json`. Pure numpy on saved JSON — no GPU, no model loading.

**Captures:** test-set sampling variance. **Does not capture:** adapter-training variance
(see the multi-seed runs).

All trajectories draw from one `default_rng(SEED)` stream in
sorted `pair_id` order, so the CIs depend on *exactly which trajectories are in the run*.
Adding or removing one shifts the draws for everything after it. The population is therefore
an explicit list of `pair_id`s, never a glob. With `POPULATION = "published"`, this notebook
reproduces `strict_overshoot_bootstrap.json` field-for-field (verified 2026-09-28, 22/22).

In [ ]:
# --- setup ------------------------------------------------------------------
import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path("/content/shared-truth")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "-b","rebuttal","https://github.com/KingTechnician/shared-truth.git",
                    str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
sys.path.insert(0, str(REPO_DIR))

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass   # not in Colab, or no secret set; public repos still work

import json, numpy as np
from shared_truth import metrics, storage
print("shared_truth @", storage.provenance()["shared_truth_sha"])

shared_truth @ e1371dd9eccf4b2e82f9f27071bba7145bfb4948


In [16]:
# --- config -----------------------------------------------------------------
SOURCE     = "hf"          # "hf" (results repo) or "drive" (mounted Drive)
POPULATION = "multiseed_full"   # a key of POPULATIONS below: "published", "multiseed_simple", "multiseed_full"
B          = 10_000
SEED       = 42
NEAR_ZERO_PP = 0.5         # flag CI lower bounds within this margin of 0
OUT_NAME   = None          # None -> strict_overshoot_bootstrap__<POPULATION>.json

## Populations

Each population is a pinned list of `pair_id`s. To add one, list it explicitly. The
`list_variants` cell below shows what's available, grouped by variant suffix.

`published` is the 22 trajectories behind the paper's bootstrap sentence. It contains one
**known duplicate**: `llama3_2-3b-instruct_l12_to_qwen2_5-1_5b-instruct_l12` appears as both
`__transfer_rsa` and `__transfer_truth`.

In [17]:
POPULATIONS = {
    "published": [
        "gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__truth",
        "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__truth",
        "llama3-8b-instruct_l12_to_gemma2-2b-instruct_l13__transfer_truth",
        "llama3-8b-instruct_l12_to_mistral-7b-instruct_l13__truth",
        "llama3-8b-instruct_l14_to_gemma2-2b-instruct_l14__transfer_rsa",
        "llama3-8b-instruct_l14_to_mistral-7b-instruct_l14__transfer_rsa",
        "llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13__truth",
        "llama3_2-3b-instruct_l12_to_mistral-7b-instruct_l13__transfer_truth",
        "llama3_2-3b-instruct_l12_to_mistral-7b-instruct_l14__transfer_rsa",
        "llama3_2-3b-instruct_l12_to_qwen2_5-1_5b-instruct_l12__transfer_rsa",
        "llama3_2-3b-instruct_l12_to_qwen2_5-1_5b-instruct_l12__transfer_truth",
        "llama3_2-3b-instruct_l12_to_qwen2_5-1_5b-instruct_l17__truth",
        "llama3_2-3b-instruct_l13_to_gemma2-2b-instruct_l14__transfer_rsa",
        "mistral-7b-instruct_l13_to_gemma-7b-instruct_l19__truth",
        "mistral-7b-instruct_l14_to_gemma-7b-instruct_l15__transfer_rsa",
        "qwen2_5-1_5b-instruct_l12_to_gemma2-2b-instruct_l11__transfer_rsa",
        "qwen2_5-1_5b-instruct_l12_to_llama3_2-3b-instruct_l12__transfer_rsa",
        "qwen2_5-1_5b-instruct_l17_to_gemma2-2b-instruct_l13__truth",
        "qwen2_5-1_5b-instruct_l17_to_llama3_2-3b-instruct_l12__truth",
        "qwen2_5-7b-instruct_l21_to_gemma-7b-instruct_l19__truth",
        "qwen2_5-7b-instruct_l21_to_llama3-8b-instruct_l12__truth",
        "qwen2_5-7b-instruct_l21_to_mistral-7b-instruct_l13__truth",
    ],
    # Multi-seed replication (appendix): the same 12 adapters (orig + seeds 42-44 on three
    # trajectories), swept on simple statements (n=1599) and on the full test set (n=2684).
    # On the simple eval, the Gemma-2-2B -> Qwen2.5-1.5B seeds 42-44 used earlier checkpoints.
    "multiseed_simple": [
        "gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__truth_origadapter",
        "gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__truth_seed42",
        "gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__truth_seed43",
        "gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__truth_seed44",
        "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__truth_origadapter",
        "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__truth_seed42",
        "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__truth_seed43",
        "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__truth_seed44",
        "llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13__truth_origadapter",
        "llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13__truth_seed42",
        "llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13__truth_seed43",
        "llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13__truth_seed44",
    ],
    "multiseed_full": [
        "gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__stmt_origadapter",
        "gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__stmt_seed42",
        "gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__stmt_seed43",
        "gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__stmt_seed44",
        "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_origadapter",
        "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_seed42",
        "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_seed43",
        "gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_seed44",
        "llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13__stmt_origadapter",
        "llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13__stmt_seed42",
        "llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13__stmt_seed43",
        "llama3_2-3b-instruct_l12_to_gemma2-2b-instruct_l13__stmt_seed44",
    ],
}

# The published file this notebook must reproduce, per population (None = no reference).
REFERENCE_FILES = {"published": "strict_overshoot_bootstrap.json"}

In [18]:
# --- resolve data root ------------------------------------------------------
if SOURCE == "hf":
    ROOT = Path(storage.fetch_results())            # sweep JSONs + root JSONs only
elif SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/shared-truth")
else:
    raise ValueError(SOURCE)
SWEEPS = ROOT / storage.RESULTS_PREFIX
print("data root:", ROOT)

def list_variants():
    '''Available trajectories grouped by variant suffix — for pinning new populations.'''
    from collections import defaultdict
    groups = defaultdict(list)
    for f in sorted(SWEEPS.glob("*/sweep_results.json")):
        pid = f.parent.name
        groups[pid.split("__", 1)[1] if "__" in pid else "(none)"].append(pid)
    for v, ids in sorted(groups.items()):
        print(f"__{v:<20} {len(ids):>3}")
    return groups

_ = list_variants()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 79 files:   0%|          | 0/79 [00:00<?, ?it/s]

data root: /root/.cache/huggingface/hub/datasets--KingTechnician--shared-truth-results/snapshots/82ed554f829e963ff45b84921083a4c513713ae8
__stmt_origadapter       3
__stmt_randbase          3
__stmt_seed42            3
__stmt_seed43            3
__stmt_seed44            3
__stmt_shufbase          3
__transfer_rsa           8
__transfer_truth         3
__truth                 11
__truth_origadapter      3
__truth_seed42           3
__truth_seed43           3
__truth_seed44           3


In [19]:
# --- load the pinned population ---------------------------------------------
ids = sorted(POPULATIONS[POPULATION])          # sorted = the order the draws were made in
if len(set(ids)) != len(ids):
    raise SystemExit("duplicate pair_id in population list")

missing = [pid for pid in ids if not (SWEEPS / pid / "sweep_results.json").exists()]
if missing:
    raise SystemExit(f"{len(missing)} missing — a partial run would shift every CI after it:\n  "
                     + "\n  ".join(missing))

payloads = {pid: json.load(open(SWEEPS / pid / "sweep_results.json")) for pid in ids}
for pid, p in payloads.items():
    if "true_scores" not in p["alpha_sweep"][0]:
        raise SystemExit(f"{pid}: sweep rows lack true/false scores; strict overshoot not computable")
print(f"{POPULATION}: {len(ids)} trajectories loaded")

multiseed_full: 12 trajectories loaded


In [20]:
# --- bootstrap --------------------------------------------------------------
import time
rng = np.random.default_rng(SEED)
results = []
for i, pid in enumerate(ids, 1):
    t0 = time.time()
    r = metrics.bootstrap_trajectory(payloads[pid]["alpha_sweep"], rng, B)
    r["pair_id"] = pid
    r["display_label"] = payloads[pid].get("display_label", pid)
    results.append(r)
    mark = "✓" if r["ci_low_pp"] > 0 else "✗"
    print(f"[{i:>2}/{len(ids)}] {mark} {r['delta_orig_pp']:+6.2f}pp "
          f"[{r['ci_low_pp']:+5.2f}, {r['ci_high_pp']:+5.2f}]  n={r['n_test']}  "
          f"{time.time()-t0:3.0f}s  {pid}", flush=True)

[ 1/12] ✓  +5.59pp [+4.47, +6.78]  n=2684   19s  gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__stmt_origadapter
[ 2/12] ✓  +4.43pp [+3.43, +5.66]  n=2684   18s  gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__stmt_seed42
[ 3/12] ✓  +3.17pp [+2.42, +4.32]  n=2684   18s  gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__stmt_seed43
[ 4/12] ✓  +2.31pp [+1.60, +3.39]  n=2684   19s  gemma2-2b-instruct_l13_to_llama3_2-3b-instruct_l12__stmt_seed44
[ 5/12] ✓  +2.38pp [+1.30, +3.46]  n=2684   19s  gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_origadapter
[ 6/12] ✗  +0.19pp [+0.00, +0.78]  n=2684   19s  gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_seed42
[ 7/12] ✗  +0.00pp [+0.00, +0.22]  n=2684   19s  gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_seed43
[ 8/12] ✗  +0.00pp [+0.00, +0.26]  n=2684   19s  gemma2-2b-instruct_l13_to_qwen2_5-1_5b-instruct_l17__stmt_seed44
[ 9/12] ✓  +2.53pp [+1.38, +3.69]  n=2684   19s  llama3_2-3b-instruct_l12_to_gemma

## Duplicate detection

Two trajectories are duplicates when their true/false score arrays are identical at every α.
The first `pair_id` in sorted order is kept as canonical.

In [14]:
def _sig(sweep):
    return tuple((tuple(r["true_scores"]), tuple(r["false_scores"])) for r in sweep)

seen, duplicates = {}, {}
for pid in ids:
    s = _sig(payloads[pid]["alpha_sweep"])
    if s in seen:
        duplicates[pid] = seen[s]
    else:
        seen[s] = pid

for dup, canon in duplicates.items():
    print(f"duplicate: {dup}\n       of: {canon}")
if not duplicates:
    print("no duplicate trajectories")

unique = [r for r in results if r["pair_id"] not in duplicates]

def summarize(rows):
    d = np.array([r["delta_orig_pp"] for r in rows])
    return {
        "n": len(rows),
        "n_positive": int((d > 0).sum()),
        "n_excludes_zero": int(sum(r["ci_low_pp"] > 0 for r in rows)),
        "median_delta_pp": float(np.median(d)),
        "iqr_pp": [float(np.percentile(d, 25)), float(np.percentile(d, 75))],
    }

S_raw, S_uniq = summarize(results), summarize(unique)
print(f"\nall rows : CI excludes 0 on {S_raw['n_excludes_zero']}/{S_raw['n']}  "
      f"median {S_raw['median_delta_pp']:+.2f}pp")
print(f"unique   : CI excludes 0 on {S_uniq['n_excludes_zero']}/{S_uniq['n']}  "
      f"median {S_uniq['median_delta_pp']:+.2f}pp  "
      f"IQR [{S_uniq['iqr_pp'][0]:+.2f}, {S_uniq['iqr_pp'][1]:+.2f}]")

near_zero = [r for r in unique if abs(r["ci_low_pp"]) < NEAR_ZERO_PP]
if near_zero:
    print(f"\n{len(near_zero)} unique trajectories have CI lower bound within {NEAR_ZERO_PP}pp of 0")

no duplicate trajectories

all rows : CI excludes 0 on 10/12  median +1.75pp
unique   : CI excludes 0 on 10/12  median +1.75pp  IQR [+1.05, +5.38]

4 unique trajectories have CI lower bound within 0.5pp of 0


In [21]:
# --- regression check against the published file ----------------------------
ref_name = REFERENCE_FILES.get(POPULATION)
if ref_name is None:
    print(f"no reference file for population {POPULATION!r}; skipping")
else:
    ref = json.load(open(ROOT / ref_name))
    FIELDS = ("delta_orig_pp", "ci_low_pp", "ci_high_pp", "median_pp", "n_test", "m1", "m0")
    o = {r["pair_id"]: r for r in ref["per_trajectory"]}
    n = {r["pair_id"]: r for r in results}
    bad = sorted(set(o) ^ set(n))
    for pid in sorted(set(o) & set(n)):
        if any(abs(float(o[pid][k]) - float(n[pid][k])) > 1e-9 for k in FIELDS):
            bad.append(pid)
    if ref["B"] != B or ref["seed"] != SEED:
        raise SystemExit(f"config differs from reference (B={ref['B']}, seed={ref['seed']})")
    if bad:
        raise SystemExit(f"REGRESSION: {len(bad)} rows differ from {ref_name}:\n  " + "\n  ".join(bad))
    print(f"✓ reproduces {ref_name} exactly ({len(o)}/{len(o)} rows, all fields)")

no reference file for population 'multiseed_full'; skipping


In [ ]:
# --- save -------------------------------------------------------------------
dup_note = (f" One trajectory appears twice in this set under two directory names "
            f"(identical scores); counting it once gives {S_uniq['n_excludes_zero']} of {S_uniq['n']}."
            if duplicates else "")
summary = (
    f"A paired stratified bootstrap (B={B:,}, seed={SEED}) over the held-out test set within "
    f"each trajectory yields per-trajectory 95\\% CIs for the strict overshoot "
    f"$\\Delta\\mathrm{{Acc}}(\\tau^*)$. The 95\\% bootstrap CI lower bound exceeds zero on "
    f"{S_uniq['n_excludes_zero']} of {S_uniq['n']} trajectories. The aggregate median is "
    f"${S_uniq['median_delta_pp']:+.2f}$~pp (IQR "
    f"$[{S_uniq['iqr_pp'][0]:+.2f}, {S_uniq['iqr_pp'][1]:+.2f}]$). This bootstrap captures "
    f"test-set sampling variance but not adapter-training variance."
)
print(summary + ("\n\n[note]" + dup_note if dup_note else ""))

out = {
    "population": POPULATION,
    "pair_ids": ids,
    "B": B, "seed": SEED,
    "summary_all_rows": S_raw,
    "summary_unique": S_uniq,
    "duplicates": duplicates,
    "near_zero_trajectories": [r["pair_id"] for r in near_zero],
    "per_trajectory": results,
    "summary_block": summary,
    "provenance": storage.provenance(),
}
out_path = Path("/content") / (OUT_NAME or f"strict_overshoot_bootstrap__{POPULATION}.json")
json.dump(out, open(out_path, "w"), indent=2)
print(f"\nsaved -> {out_path}")

A paired stratified bootstrap (B=10,000, seed=42) over the held-out test set within each trajectory yields per-trajectory 95\% CIs for the strict overshoot $\Delta\mathrm{Acc}(\tau^*)$. The 95\% bootstrap CI lower bound exceeds zero on 17 of 21 trajectories. The aggregate median is $+1.13$~pp (IQR $[+0.50, +2.88]$). This bootstrap captures test-set sampling variance but not adapter-training variance.

[note] One trajectory appears twice in this set under two directory names (identical scores); counting it once gives 17 of 21.

saved -> /content/strict_overshoot_bootstrap__published.json
